# songo-model-stockfish-for-google-collab - All In One

Notebook unique pour:
- monter Google Drive
- cloner ou mettre a jour le code depuis GitHub
- installer les dependances
- lancer la generation de dataset
- construire le dataset final
- entrainer le modele
- evaluer le modele
- benchmarker le modele
- reprendre un job interrompu avec le meme `job_id`


## 1. Monter Drive

In [11]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configuration du projet

In [12]:
import os
import sys
from pathlib import Path

GIT_REPO_URL = 'https://github.com/GlennEriss/songo-model-stockfish-for-google-collab.git'
GIT_BRANCH = 'main'
PROJECT_NAME = 'songo-model-stockfish-for-google-collab'
DRIVE_ROOT = '/content/drive/MyDrive/songo-stockfish'
DEFAULT_WORKTREE = f'/content/{PROJECT_NAME}'

detected_worktree = ''
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'src' / 'songo_model_stockfish').exists() and (candidate / 'config').exists():
        detected_worktree = str(candidate)
        break

WORKTREE = os.environ.get('WORKTREE') or detected_worktree or DEFAULT_WORKTREE
PYTHON_BIN = sys.executable or 'python3'

print('GIT_REPO_URL =', GIT_REPO_URL)
print('GIT_BRANCH   =', GIT_BRANCH)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('WORKTREE     =', WORKTREE)
print('PYTHON_BIN   =', PYTHON_BIN)


GIT_REPO_URL = https://github.com/GlennEriss/songo-model-stockfish-for-google-collab.git
GIT_BRANCH   = main
DRIVE_ROOT   = /content/drive/MyDrive/songo-stockfish
WORKTREE     = /content/songo-model-stockfish-for-google-collab


## 3. Initialiser Drive

In [13]:
import os

for path in [
    f'{DRIVE_ROOT}/code',
    f'{DRIVE_ROOT}/code_snapshots',
    f'{DRIVE_ROOT}/data/raw',
    f'{DRIVE_ROOT}/data/raw_colab_pro',
    f'{DRIVE_ROOT}/data/raw_full_matrix_colab_pro',
    f'{DRIVE_ROOT}/data/sampled',
    f'{DRIVE_ROOT}/data/sampled_colab_pro',
    f'{DRIVE_ROOT}/data/sampled_full_matrix_colab_pro',
    f'{DRIVE_ROOT}/data/datasets',
    f'{DRIVE_ROOT}/jobs',
    f'{DRIVE_ROOT}/models/checkpoints',
    f'{DRIVE_ROOT}/models/final',
    f'{DRIVE_ROOT}/models/promoted/best',
    f'{DRIVE_ROOT}/models/lineage',
    f'{DRIVE_ROOT}/logs',
    f'{DRIVE_ROOT}/reports/evaluations',
    f'{DRIVE_ROOT}/reports/benchmarks',
    f'{DRIVE_ROOT}/exports',
]:
    os.makedirs(path, exist_ok=True)

print('Drive layout ready')


Drive layout ready


## 4. Cloner ou mettre a jour le repo

In [14]:
import os
import shutil
import subprocess

if not os.path.exists(os.path.join(WORKTREE, '.git')):
    if os.path.exists(WORKTREE):
        shutil.rmtree(WORKTREE)
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, GIT_REPO_URL, WORKTREE], check=True)
else:
    subprocess.run(['git', '-C', WORKTREE, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', WORKTREE, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', WORKTREE, 'pull', '--ff-only', 'origin', GIT_BRANCH], check=True)

print('Worktree ready')


Worktree ready


## 5. Installer les dependances

In [6]:
import subprocess
subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-r', f'{WORKTREE}/requirements.txt'], check=True)
print('Environment ready in Colab runtime')


Environment ready in Colab runtime


## 6. Choisir les configs et job ids

Le pipeline dataset supporte maintenant plusieurs modes de generation:
- `benchmatch`: generation longue a partir des matchs de reference
- `clone_existing`: creation d'une nouvelle source de dataset a partir d'un corpus deja existant
- `derive_existing`: creation d'une nouvelle variante filtree a partir d'un corpus existant
- `augment_existing`: creation d'une nouvelle grande source en rejouant des coups legaux a partir d'un corpus existant, avec deduplication des positions
- `merge_existing`: fusion de plusieurs sources deja versionnees en une nouvelle grande source

Ensuite `dataset-build` peut choisir explicitement quelle source de dataset il enrichit.
Les originaux restent conserves, car chaque source et chaque dataset construit sont versionnes separement.
Tu peux aussi forcer un nouvel identifiant de dataset construit avec `DATASET_BUILD_ID_OVERRIDE`.


Pour comparer proprement deux familles de modeles, garde le meme dataset et change seulement `TRAIN_CONFIG` et les `JOB_ID`.

Exemple recommande:
- stable: `TRAIN_CONFIG = 'config/train.full_matrix.colab_pro.yaml'`
- residual: `TRAIN_CONFIG = 'config/train.full_matrix.colab_pro.residual.yaml'`

Important:
- utilise des `TRAIN_JOB_ID`, `EVALUATION_JOB_ID` et `BENCHMARK_JOB_ID` differents entre stable et residual
- ne reutilise pas le meme `TRAIN_JOB_ID` pour deux architectures differentes


In [ ]:
DATASET_GENERATE_CONFIG = 'config/dataset_generation.full_matrix.colab_pro.yaml'
DATASET_GENERATE_CONFIG_ACTIVE = DATASET_GENERATE_CONFIG
DATASET_BUILD_CONFIG = 'config/dataset_build.full_matrix.colab_pro.yaml'
DATASET_MERGE_FINAL_CONFIG = 'config/dataset_merge_final.colab_pro.yaml'
TRAIN_CONFIG = 'config/train.full_matrix.colab_pro.yaml'
TRAIN_CONTINUE_CONFIG = 'config/train.full_matrix.colab_pro.yaml'
TRAIN_SCRATCH_CONFIG = 'config/train.full_matrix.colab_pro.scratch.yaml'
# Variante experimentale possible:
# TRAIN_CONFIG = 'config/train.full_matrix.colab_pro.residual.yaml'
EVALUATION_CONFIG = 'config/evaluation.full_matrix.colab_pro.yaml'
BENCHMARK_CONFIG = 'config/benchmark.colab_pro.yaml'
TRAIN_CONFIG_ACTIVE = TRAIN_CONFIG
TRAIN_CONTINUE_CONFIG_ACTIVE = TRAIN_CONTINUE_CONFIG
TRAIN_SCRATCH_CONFIG_ACTIVE = TRAIN_SCRATCH_CONFIG
EVALUATION_CONFIG_ACTIVE = EVALUATION_CONFIG
BENCHMARK_CONFIG_ACTIVE = BENCHMARK_CONFIG

DATASET_GENERATE_JOB_ID = ''
DATASET_BUILD_JOB_ID = ''
DATASET_MERGE_FINAL_JOB_ID = 'dataset_merge_final_colab_pro_001'
TRAIN_JOB_ID = 'train_colab_pro_stable_001'
TRAIN_CONTINUE_JOB_ID = 'train_colab_pro_continue_001'
TRAIN_SCRATCH_JOB_ID = 'train_colab_pro_scratch_001'
TRAIN_CONTINUE_PARTIAL_JOB_ID = 'train_colab_pro_continue_partial_001'
TRAIN_SCRATCH_PARTIAL_JOB_ID = 'train_colab_pro_scratch_partial_001'
EVALUATION_JOB_ID = 'eval_colab_pro_stable_001'
BENCHMARK_JOB_ID = 'benchmark_colab_pro_stable_001'


In [ ]:
DATASET_GENERATION_MODE = 'benchmatch'
DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro_bench_models_20m'
SOURCE_DATASET_ID_FOR_GENERATION = ''
SOURCE_DATASET_IDS_FOR_GENERATION = []
DERIVATION_STRATEGY = 'unique_positions'
AUGMENT_INCLUDE_ORIGINAL_SAMPLES = True
AUGMENT_MAX_DEPTH = 2
AUGMENT_MAX_BRANCHING = 3
AUGMENT_MAX_GENERATED_PER_SOURCE_SAMPLE = 8
TARGET_SAMPLES = 20000000
BENCHMATCH_GAMES = 400
BENCHMATCH_CLASSIC_AGENTS = [
    'minimax:medium',
    'minimax:hard',
    'mcts:medium',
    'mcts:hard',
]
BENCHMATCH_INCLUDE_ALL_REGISTERED_MODELS = True
BENCHMATCH_MODEL_PREFIX = 'songo_policy_value_colab_pro_'
BENCHMATCH_MODEL_IDS = []
BENCHMATCH_EXCLUDE_MODEL_IDS = []
BENCHMATCH_MODEL_LIMIT = 0
BENCHMATCH_INCLUDE_SELF_PLAY = True
BENCHMATCH_ORDERED_MATCHUPS = True
BENCHMATCH_MODEL_AGENT_DEVICE = 'cpu'
DATASET_GENERATE_ACTIVE_CONFIG_PATH = f'{WORKTREE}/config/generated/dataset_generation.full_matrix.colab_pro.active.yaml'
SOURCE_DATASET_ID_FOR_BUILD = DATASET_SOURCE_ID
DATASET_BUILD_ID_OVERRIDE = 'dataset_v4_full_matrix_colab_pro_insane_20m'
TARGET_LABELED_SAMPLES = 20000000
PARTIAL_TRAIN_DATASET_ID = 'dataset_v3_full_matrix_colab_pro_insane_10m'
MERGED_FINAL_DATASET_ID = 'dataset_merged_final_colab_pro_insane_1m'
MERGE_ALL_BUILT_DATASETS = True
MERGE_SOURCE_DATASET_IDS = []
COMPARE_DATASET_ID_A = ''
COMPARE_DATASET_ID_B = ''
SELECTED_MODEL_ID = ''
CPU_SAFE_NUM_WORKERS = 2
CPU_SAFE_PREFETCH_FACTOR = 2
TRAIN_CONTINUE_PARTIAL_CONFIG_PATH = f'{WORKTREE}/config/generated/train.full_matrix.colab_pro.partial.yaml'
TRAIN_SCRATCH_PARTIAL_CONFIG_PATH = f'{WORKTREE}/config/generated/train.full_matrix.colab_pro.scratch.partial.yaml'
TRAIN_CONTINUE_CPU_CONFIG_PATH = f'{WORKTREE}/config/generated/train.full_matrix.colab_pro.cpu.yaml'
TRAIN_SCRATCH_CPU_CONFIG_PATH = f'{WORKTREE}/config/generated/train.full_matrix.colab_pro.scratch.cpu.yaml'
EVALUATION_CPU_CONFIG_PATH = f'{WORKTREE}/config/generated/evaluation.full_matrix.colab_pro.cpu.yaml'
BENCHMARK_CPU_CONFIG_PATH = f'{WORKTREE}/config/generated/benchmark.colab_pro.cpu.yaml'

def _auto_dataset_generate_job_id(mode: str, dataset_source_id: str, target_samples: int) -> str:
    safe_source = dataset_source_id.replace('-', '_')
    if mode == 'benchmatch':
        return f"dataset_benchmatch_{safe_source}_{target_samples}_001"
    if mode == 'clone_existing':
        return f"dataset_clone_{safe_source}_{target_samples}_001"
    if mode == 'derive_existing':
        return f"dataset_derive_{safe_source}_{target_samples}_001"
    if mode == 'augment_existing':
        return f"dataset_aug_{safe_source}_{target_samples}_001"
    if mode == 'merge_existing':
        return f"dataset_merge_{safe_source}_{target_samples}_001"
    return f"dataset_generate_{safe_source}_{target_samples}_001"

if not DATASET_GENERATE_JOB_ID:
    DATASET_GENERATE_JOB_ID = _auto_dataset_generate_job_id(DATASET_GENERATION_MODE, DATASET_SOURCE_ID, TARGET_SAMPLES)

def _auto_dataset_build_job_id(dataset_id_override: str, target_labeled_samples: int) -> str:
    if dataset_id_override:
        safe_dataset = dataset_id_override.replace('-', '_')
    else:
        safe_dataset = 'dataset_build'
    return f"build_{safe_dataset}_{target_labeled_samples}_001"

if not DATASET_BUILD_JOB_ID:
    DATASET_BUILD_JOB_ID = _auto_dataset_build_job_id(DATASET_BUILD_ID_OVERRIDE, TARGET_LABELED_SAMPLES)

DATASET_GENERATE_EXTRA_ARGS = f"--generation-mode {DATASET_GENERATION_MODE} --dataset-source-id {DATASET_SOURCE_ID}"
if SOURCE_DATASET_ID_FOR_GENERATION:
    DATASET_GENERATE_EXTRA_ARGS += f" --source-dataset-id {SOURCE_DATASET_ID_FOR_GENERATION}"
if DATASET_GENERATION_MODE == 'merge_existing' and SOURCE_DATASET_IDS_FOR_GENERATION:
    joined_source_ids = ' '.join(SOURCE_DATASET_IDS_FOR_GENERATION)
    DATASET_GENERATE_EXTRA_ARGS += f" --source-dataset-ids {joined_source_ids}"
if DATASET_GENERATION_MODE == 'derive_existing':
    DATASET_GENERATE_EXTRA_ARGS += f" --derivation-strategy {DERIVATION_STRATEGY}"
if DATASET_GENERATION_MODE == 'augment_existing':
    if AUGMENT_INCLUDE_ORIGINAL_SAMPLES:
        DATASET_GENERATE_EXTRA_ARGS += ' --augmentation-include-original-samples'
    else:
        DATASET_GENERATE_EXTRA_ARGS += ' --augmentation-only-new-samples'
    DATASET_GENERATE_EXTRA_ARGS += f" --augmentation-max-depth {AUGMENT_MAX_DEPTH}"
    DATASET_GENERATE_EXTRA_ARGS += f" --augmentation-max-branching {AUGMENT_MAX_BRANCHING}"
    DATASET_GENERATE_EXTRA_ARGS += f" --augmentation-max-generated-per-source-sample {AUGMENT_MAX_GENERATED_PER_SOURCE_SAMPLE}"
if TARGET_SAMPLES:
    DATASET_GENERATE_EXTRA_ARGS += f" --target-samples {TARGET_SAMPLES}"

DATASET_BUILD_EXTRA_ARGS = ''
if SOURCE_DATASET_ID_FOR_BUILD:
    DATASET_BUILD_EXTRA_ARGS += f" --source-dataset-id {SOURCE_DATASET_ID_FOR_BUILD}"
if DATASET_BUILD_ID_OVERRIDE:
    DATASET_BUILD_EXTRA_ARGS += f" --dataset-id-override {DATASET_BUILD_ID_OVERRIDE}"
if TARGET_LABELED_SAMPLES:
    DATASET_BUILD_EXTRA_ARGS += f" --target-labeled-samples {TARGET_LABELED_SAMPLES}"

DATASET_MERGE_FINAL_EXTRA_ARGS = f"--dataset-id {MERGED_FINAL_DATASET_ID}"
if MERGE_ALL_BUILT_DATASETS:
    DATASET_MERGE_FINAL_EXTRA_ARGS += ' --include-all-built'
elif MERGE_SOURCE_DATASET_IDS:
    joined_ids = ' '.join(MERGE_SOURCE_DATASET_IDS)
    DATASET_MERGE_FINAL_EXTRA_ARGS += f" --source-dataset-ids {joined_ids}"

print('DATASET_GENERATION_MODE          =', DATASET_GENERATION_MODE)
print('DATASET_SOURCE_ID                =', DATASET_SOURCE_ID)
print('TARGET_SAMPLES                   =', TARGET_SAMPLES)
print('BENCHMATCH_GAMES                 =', BENCHMATCH_GAMES)
print('BENCHMATCH_CLASSIC_AGENTS        =', BENCHMATCH_CLASSIC_AGENTS)
print('BENCHMATCH_INCLUDE_ALL_REGISTERED_MODELS =', BENCHMATCH_INCLUDE_ALL_REGISTERED_MODELS)
print('BENCHMATCH_MODEL_PREFIX          =', BENCHMATCH_MODEL_PREFIX)
print('BENCHMATCH_MODEL_IDS             =', BENCHMATCH_MODEL_IDS)
print('BENCHMATCH_EXCLUDE_MODEL_IDS     =', BENCHMATCH_EXCLUDE_MODEL_IDS)
print('BENCHMATCH_MODEL_LIMIT           =', BENCHMATCH_MODEL_LIMIT)
print('BENCHMATCH_INCLUDE_SELF_PLAY     =', BENCHMATCH_INCLUDE_SELF_PLAY)
print('BENCHMATCH_ORDERED_MATCHUPS      =', BENCHMATCH_ORDERED_MATCHUPS)
print('BENCHMATCH_MODEL_AGENT_DEVICE    =', BENCHMATCH_MODEL_AGENT_DEVICE)
print('DATASET_GENERATE_ACTIVE_CONFIG_PATH =', DATASET_GENERATE_ACTIVE_CONFIG_PATH)
print('SOURCE_DATASET_ID_FOR_BUILD      =', SOURCE_DATASET_ID_FOR_BUILD)
print('DATASET_BUILD_ID_OVERRIDE        =', DATASET_BUILD_ID_OVERRIDE)
print('TARGET_LABELED_SAMPLES           =', TARGET_LABELED_SAMPLES)
print('PARTIAL_TRAIN_DATASET_ID         =', PARTIAL_TRAIN_DATASET_ID)
print('CPU_SAFE_NUM_WORKERS             =', CPU_SAFE_NUM_WORKERS)
print('CPU_SAFE_PREFETCH_FACTOR         =', CPU_SAFE_PREFETCH_FACTOR)
print('DATASET_BUILD_JOB_ID             =', DATASET_BUILD_JOB_ID)
print('MERGED_FINAL_DATASET_ID          =', MERGED_FINAL_DATASET_ID)
print('MERGE_ALL_BUILT_DATASETS         =', MERGE_ALL_BUILT_DATASETS)
print('MERGE_SOURCE_DATASET_IDS         =', MERGE_SOURCE_DATASET_IDS)
print('COMPARE_DATASET_ID_A             =', COMPARE_DATASET_ID_A)
print('COMPARE_DATASET_ID_B             =', COMPARE_DATASET_ID_B)
print('SELECTED_MODEL_ID                =', SELECTED_MODEL_ID or '<latest>')
print('DATASET_GENERATE_JOB_ID          =', DATASET_GENERATE_JOB_ID)
if DATASET_GENERATION_MODE != 'benchmatch' and 'benchmatch' in DATASET_GENERATE_JOB_ID:
    print('WARNING: le job_id courant contient encore benchmatch; change-le ou laisse DATASET_GENERATE_JOB_ID vide pour utiliser l'id auto')


Bascule runtime automatique:
- si `torch.cuda.is_available() == True`, le notebook garde les configs GPU normales
- sinon il genere automatiquement des configs CPU-safe et les commandes train/eval/benchmark les utilisent


In [ ]:
from pathlib import Path

try:
    import torch
except ModuleNotFoundError:
    torch = None

try:
    import yaml
except ModuleNotFoundError:
    yaml = None

RUNTIME_HAS_CUDA = bool(torch is not None and torch.cuda.is_available())
TRAIN_CONFIG_ACTIVE = TRAIN_CONFIG
TRAIN_CONTINUE_CONFIG_ACTIVE = TRAIN_CONTINUE_CONFIG
TRAIN_SCRATCH_CONFIG_ACTIVE = TRAIN_SCRATCH_CONFIG
EVALUATION_CONFIG_ACTIVE = EVALUATION_CONFIG
BENCHMARK_CONFIG_ACTIVE = BENCHMARK_CONFIG

def _make_cpu_safe_runtime_cfg(base_cfg: dict):
    runtime_cfg = dict(base_cfg.get('runtime', {}) or {})
    runtime_cfg['device'] = 'cpu'
    runtime_cfg['num_workers'] = int(CPU_SAFE_NUM_WORKERS)
    runtime_cfg['pin_memory'] = False
    runtime_cfg['persistent_workers'] = bool(CPU_SAFE_NUM_WORKERS > 0)
    runtime_cfg['prefetch_factor'] = int(CPU_SAFE_PREFETCH_FACTOR) if CPU_SAFE_NUM_WORKERS > 0 else 0
    runtime_cfg['mixed_precision'] = False
    base_cfg['runtime'] = runtime_cfg
    return base_cfg

def _write_cpu_safe_config(base_relative_path: str, target_path_str: str):
    if yaml is None:
        raise RuntimeError('PyYAML introuvable')
    base_path = Path(WORKTREE) / base_relative_path
    target_path = Path(target_path_str)
    cfg = yaml.safe_load(base_path.read_text(encoding='utf-8')) or {}
    cfg = _make_cpu_safe_runtime_cfg(cfg)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    target_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
    return target_path

if RUNTIME_HAS_CUDA:
    print('Runtime GPU detecte: les configs GPU standards restent actives')
else:
    if yaml is None:
        print('PyYAML introuvable: impossible de generer les configs CPU-safe automatiquement')
    else:
        TRAIN_CONTINUE_CONFIG_ACTIVE = str(_write_cpu_safe_config(TRAIN_CONTINUE_CONFIG, TRAIN_CONTINUE_CPU_CONFIG_PATH))
        TRAIN_SCRATCH_CONFIG_ACTIVE = str(_write_cpu_safe_config(TRAIN_SCRATCH_CONFIG, TRAIN_SCRATCH_CPU_CONFIG_PATH))
        EVALUATION_CONFIG_ACTIVE = str(_write_cpu_safe_config(EVALUATION_CONFIG, EVALUATION_CPU_CONFIG_PATH))
        BENCHMARK_CONFIG_ACTIVE = str(_write_cpu_safe_config(BENCHMARK_CONFIG, BENCHMARK_CPU_CONFIG_PATH))
        TRAIN_CONFIG_ACTIVE = TRAIN_CONTINUE_CONFIG_ACTIVE
        print('Runtime CPU detecte: configs CPU-safe generees')

print('RUNTIME_HAS_CUDA            =', RUNTIME_HAS_CUDA)
print('TRAIN_CONTINUE_CONFIG_ACTIVE =', TRAIN_CONTINUE_CONFIG_ACTIVE)
print('TRAIN_SCRATCH_CONFIG_ACTIVE  =', TRAIN_SCRATCH_CONFIG_ACTIVE)
print('EVALUATION_CONFIG_ACTIVE     =', EVALUATION_CONFIG_ACTIVE)
print('BENCHMARK_CONFIG_ACTIVE      =', BENCHMARK_CONFIG_ACTIVE)


Presets utiles:

- Corpus long de reference par matchs:
  - `DATASET_GENERATION_MODE = 'benchmatch'`
  - `DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro'`
  - `SOURCE_DATASET_ID_FOR_GENERATION = ''`

- Nouvelle source rapide a partir du corpus 1M deja existant:
  - `DATASET_GENERATION_MODE = 'clone_existing'`
  - `DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro_variant_a'`
  - `SOURCE_DATASET_ID_FOR_GENERATION = 'sampled_full_matrix_colab_pro'`

- Variante derivee sans relancer les matchs:
  - `DATASET_GENERATION_MODE = 'derive_existing'`
  - `DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro_unique'`
  - `SOURCE_DATASET_ID_FOR_GENERATION = 'sampled_full_matrix_colab_pro'`
  - `DERIVATION_STRATEGY = 'unique_positions'`
  - `TARGET_SAMPLES = 2000000`

- Grande source augmentee sans relancer les matchs:
  - `DATASET_GENERATION_MODE = 'augment_existing'`
  - `DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro_aug_10m'`
  - `SOURCE_DATASET_ID_FOR_GENERATION = 'sampled_full_matrix_colab_pro'`
  - `TARGET_SAMPLES = 10000000`
  - `AUGMENT_INCLUDE_ORIGINAL_SAMPLES = True`
  - `AUGMENT_MAX_DEPTH = 2`
  - `AUGMENT_MAX_BRANCHING = 3`
  - `AUGMENT_MAX_GENERATED_PER_SOURCE_SAMPLE = 8`

- Grande source fusionnee a partir de plusieurs sources deja versionnees:
  - `DATASET_GENERATION_MODE = 'merge_existing'`
  - `DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro_giant'`
  - `SOURCE_DATASET_IDS_FOR_GENERATION = ['sampled_full_matrix_colab_pro', 'sampled_full_matrix_colab_pro_unique']`
  - `TARGET_SAMPLES = 2000000`

- Build d'un nouveau dataset final 10M a partir de la source augmentee:
  - `SOURCE_DATASET_ID_FOR_BUILD = 'sampled_full_matrix_colab_pro_aug_10m'`
  - `DATASET_BUILD_ID_OVERRIDE = 'dataset_v3_full_matrix_colab_pro_insane_10m'`
  - `TARGET_LABELED_SAMPLES = 10000000`

- Fusionner tous les datasets finaux existants en un nouveau dataset final versionne:
  - `MERGED_FINAL_DATASET_ID = 'dataset_merged_final_colab_pro_insane_1m'`
  - `MERGE_ALL_BUILT_DATASETS = True`

- Comparer deux datasets finaux existants:
  - `COMPARE_DATASET_ID_A = 'dataset_v2_full_matrix_colab_pro_insane_1m'`
  - `COMPARE_DATASET_ID_B = 'dataset_merged_final_colab_pro_insane_1m'`

- Forcer un modele precis dans la vue unifiee:
  - `SELECTED_MODEL_ID = 'songo_policy_value_colab_pro_v12'`
  - laisser `''` pour prendre automatiquement le dernier modele du registre


Convention recommandee de nommage:
- sources: `sampled_full_matrix_colab_pro_<variante>`
- datasets finaux: `dataset_vN_full_matrix_colab_pro_<variante>_insane_1m`

Evite de reutiliser un identifiant pour un contenu different.


In [ ]:
from pathlib import Path
import json

registry_path = Path(DRIVE_ROOT) / 'data' / 'dataset_registry.json'
if registry_path.exists():
    registry = json.loads(registry_path.read_text(encoding='utf-8'))
    print('Dataset sources:')
    for item in registry.get('dataset_sources', [])[-10:]:
        print('-', item.get('dataset_source_id'), '| mode=', item.get('source_mode'), '| parent=', item.get('source_dataset_id', ''), '| parents=', item.get('source_dataset_ids', []), '| samples=', item.get('sampled_positions'))
    print('\nBuilt datasets:')
    for item in registry.get('built_datasets', [])[-10:]:
        print('-', item.get('dataset_id'), '| build_mode=', item.get('build_mode', 'teacher_label'), '| source=', item.get('source_dataset_id'), '| teacher=', f"{item.get('teacher_engine')}:{item.get('teacher_level')}", '| labeled=', item.get('labeled_samples'))
else:
    print('Aucun dataset_registry.json trouve pour le moment')


Tu peux aussi utiliser directement la CLI `dataset-list` pour relire le registre avec le meme comportement que hors notebook.

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=$WORKTREE/src $PYTHON_BIN -m songo_model_stockfish.cli.main dataset-list --config $DATASET_GENERATE_CONFIG --kind all"

Cellule optionnelle pour supprimer les datasets finaux actuels devenus invalides. Elle supprime les dossiers `data/datasets/<dataset_id>` et `data/label_cache/<dataset_id>`, puis nettoie aussi le registre `built_datasets`.

In [ ]:
from pathlib import Path
import json
import shutil

DELETE_BUILT_DATASET_IDS = []
# Exemple:
# DELETE_BUILT_DATASET_IDS = [
#     'dataset_v2_full_matrix_colab_pro_insane_1m',
#     'dataset_merged_final_colab_pro_insane_1m',
# ]

registry_path = Path(DRIVE_ROOT) / 'data' / 'dataset_registry.json'
registry = json.loads(registry_path.read_text(encoding='utf-8')) if registry_path.exists() else {'dataset_sources': [], 'built_datasets': []}
built = list(registry.get('built_datasets', []))

if not DELETE_BUILT_DATASET_IDS:
    print('Aucune suppression lancee. Renseigne DELETE_BUILT_DATASET_IDS puis relance la cellule.')
else:
    deleted = []
    kept = []
    for item in built:
        dataset_id = str(item.get('dataset_id', ''))
        if dataset_id not in DELETE_BUILT_DATASET_IDS:
            kept.append(item)
            continue

        output_dir = Path(str(item.get('output_dir', ''))) if str(item.get('output_dir', '')).strip() else None
        label_cache_dir = Path(str(item.get('label_cache_dir', ''))) if str(item.get('label_cache_dir', '')).strip() else None
        if output_dir and output_dir.exists():
            shutil.rmtree(output_dir)
        if label_cache_dir and label_cache_dir.exists():
            shutil.rmtree(label_cache_dir)
        deleted.append(dataset_id)

    registry['built_datasets'] = kept
    registry_path.parent.mkdir(parents=True, exist_ok=True)
    registry_path.write_text(json.dumps(registry, indent=2, ensure_ascii=True), encoding='utf-8')
    print('Datasets finaux supprimes:', deleted)


Afficher les sources et datasets finaux encore en statut `partial`, puis les nettoyer si tu as deja une version complete.

In [ ]:
from pathlib import Path
import json

registry_path = Path(DRIVE_ROOT) / 'data' / 'dataset_registry.json'
registry = json.loads(registry_path.read_text(encoding='utf-8')) if registry_path.exists() else {'dataset_sources': [], 'built_datasets': []}

partial_sources = [item for item in registry.get('dataset_sources', []) if str(item.get('source_status', 'completed')) == 'partial']
partial_built = [item for item in registry.get('built_datasets', []) if str(item.get('build_status', 'completed')) == 'partial']

print('Partial dataset sources:')
if partial_sources:
    for item in partial_sources:
        print('-', item.get('dataset_source_id'), '| mode=', item.get('source_mode'), '| samples=', item.get('sampled_positions'), '| updated_at=', item.get('updated_at'))
else:
    print('- none')

print('\nPartial built datasets:')
if partial_built:
    for item in partial_built:
        print('-', item.get('dataset_id'), '| labeled=', item.get('labeled_samples'), '| schema=', item.get('feature_schema_version', ''), '| updated_at=', item.get('updated_at'))
else:
    print('- none')


Cellule optionnelle pour nettoyer les entrees `partial` restantes une fois qu'un dataset complet existe deja. Par defaut, elle nettoie seulement le registre. Active `DELETE_PARTIAL_FILES = True` pour supprimer aussi les dossiers associes.

In [ ]:
from pathlib import Path
import json
import shutil

CLEAN_PARTIAL_SOURCE_IDS = []
CLEAN_PARTIAL_DATASET_IDS = []
DELETE_PARTIAL_FILES = False

registry_path = Path(DRIVE_ROOT) / 'data' / 'dataset_registry.json'
registry = json.loads(registry_path.read_text(encoding='utf-8')) if registry_path.exists() else {'dataset_sources': [], 'built_datasets': []}

deleted_sources = []
kept_sources = []
for item in registry.get('dataset_sources', []):
    dataset_source_id = str(item.get('dataset_source_id', ''))
    if dataset_source_id not in CLEAN_PARTIAL_SOURCE_IDS:
        kept_sources.append(item)
        continue
    if DELETE_PARTIAL_FILES:
        for key in ('raw_dir', 'sampled_dir'):
            path_value = str(item.get(key, '')).strip()
            if path_value:
                path = Path(path_value)
                if path.exists():
                    shutil.rmtree(path)
    deleted_sources.append(dataset_source_id)

deleted_built = []
kept_built = []
for item in registry.get('built_datasets', []):
    dataset_id = str(item.get('dataset_id', ''))
    if dataset_id not in CLEAN_PARTIAL_DATASET_IDS:
        kept_built.append(item)
        continue
    if DELETE_PARTIAL_FILES:
        for key in ('output_dir', 'label_cache_dir'):
            path_value = str(item.get(key, '')).strip()
            if path_value:
                path = Path(path_value)
                if path.exists():
                    shutil.rmtree(path)
    deleted_built.append(dataset_id)

registry['dataset_sources'] = kept_sources
registry['built_datasets'] = kept_built
registry_path.parent.mkdir(parents=True, exist_ok=True)
registry_path.write_text(json.dumps(registry, indent=2, ensure_ascii=True), encoding='utf-8')

print('Partial sources removed from registry:', deleted_sources)
print('Partial built datasets removed from registry:', deleted_built)
print('DELETE_PARTIAL_FILES =', DELETE_PARTIAL_FILES)


Scanner automatiquement les anciens datasets finaux invalides. La cellule cherche ceux qui n'ont pas le nouveau schema tactique complet dans `dataset_metadata.json` ou directement dans les fichiers `.npz`.

In [ ]:
from pathlib import Path
import json
import numpy as np

registry_path = Path(DRIVE_ROOT) / 'data' / 'dataset_registry.json'
registry = json.loads(registry_path.read_text(encoding='utf-8')) if registry_path.exists() else {'built_datasets': []}
invalid_dataset_ids = []

required_npz_keys = {
    'x',
    'legal_mask',
    'policy_index',
    'policy_target_full',
    'value_target',
    'capture_move_mask',
    'safe_move_mask',
    'risky_move_mask',
    'sample_ids',
    'game_ids',
}

for item in registry.get('built_datasets', []):
    dataset_id = str(item.get('dataset_id', ''))
    output_dir = Path(str(item.get('output_dir', '')))
    metadata_path = output_dir / 'dataset_metadata.json'
    metadata = json.loads(metadata_path.read_text(encoding='utf-8')) if metadata_path.exists() else {}
    feature_schema_version = str(metadata.get('feature_schema_version', ''))
    has_policy_target_full = bool(metadata.get('has_policy_target_full', False))
    has_tactical_features = bool(metadata.get('has_tactical_features', False))
    has_tactical_move_masks = bool(metadata.get('has_tactical_move_masks', False))

    npz_ok = True
    for split_name in ('train', 'validation', 'test'):
        split_path = output_dir / f'{split_name}.npz'
        if not split_path.exists():
            continue
        with np.load(split_path, allow_pickle=True) as data:
            keys = set(data.files)
        if not required_npz_keys.issubset(keys):
            npz_ok = False
            break

    is_valid = (
        feature_schema_version == 'policy_value_tactical_v2'
        and has_policy_target_full
        and has_tactical_features
        and has_tactical_move_masks
        and npz_ok
    )
    if not is_valid:
        invalid_dataset_ids.append(dataset_id)
        print('-', dataset_id, '| invalid tactical schema | output_dir=', output_dir)

if not invalid_dataset_ids:
    print('Aucun ancien dataset final invalide detecte')
else:
    print('\nDELETE_BUILT_DATASET_IDS =', invalid_dataset_ids)


## 7. Lancer la generation du dataset

Le notebook peut maintenant generer une config benchmatch active qui combine:
- les agents classiques `minimax:*` et `mcts:*`
- les modeles du registre sous la forme `model:<model_id>`

Important:
- il n'y a pas de menu interactif au lancement
- le mode reel depend de `DATASET_GENERATION_MODE`
- en mode `benchmatch`, la cellule juste en dessous fabrique une config active avec les matchups selectionnes
- laisse `DATASET_GENERATE_JOB_ID = ""` pour que le notebook genere automatiquement un `job_id` coherent avec le mode choisi
- les parties terminees ecrivent tout de suite leur `.json` brut et leur `.jsonl` sampled
- le registre dataset et le resume partiel avancent maintenant regulierement pendant le run
- pour les modes non `benchmatch`, les dossiers `raw` et `sampled` restent isoles automatiquement a partir de `DATASET_SOURCE_ID`


In [ ]:
from pathlib import Path
import json
import re

try:
    import yaml
except ModuleNotFoundError:
    yaml = None

DATASET_GENERATE_CONFIG_ACTIVE = DATASET_GENERATE_CONFIG

if DATASET_GENERATION_MODE != 'benchmatch':
    print('Generation mode != benchmatch: la config dataset-generate de base reste active')
else:
    if yaml is None:
        print('PyYAML introuvable: impossible de generer la config benchmatch active automatiquement')
    else:
        base_path = Path(DATASET_GENERATE_CONFIG)
        if not base_path.is_absolute():
            base_path = Path(WORKTREE) / DATASET_GENERATE_CONFIG
        target_path = Path(DATASET_GENERATE_ACTIVE_CONFIG_PATH)
        cfg = yaml.safe_load(base_path.read_text(encoding='utf-8')) or {}
        dataset_cfg = dict(cfg.get('dataset_generation', {}) or {})

        registry_path = Path(DRIVE_ROOT) / 'models' / 'model_registry.json'
        registry = json.loads(registry_path.read_text(encoding='utf-8')) if registry_path.exists() else {'models': []}
        selected_model_ids = []
        seen_model_ids = set()

        def _maybe_add_model(model_id: str):
            model_id = str(model_id).strip()
            if not model_id:
                return
            if BENCHMATCH_MODEL_PREFIX and not model_id.startswith(BENCHMATCH_MODEL_PREFIX):
                return
            if model_id in BENCHMATCH_EXCLUDE_MODEL_IDS:
                return
            if model_id in seen_model_ids:
                return
            seen_model_ids.add(model_id)
            selected_model_ids.append(model_id)

        def _version_sort_key(model_id: str):
            match = re.search(r'_v(\d+)$', model_id)
            return (0, int(match.group(1))) if match else (1, model_id)

        if BENCHMATCH_INCLUDE_ALL_REGISTERED_MODELS:
            discovered = [str(item.get('model_id', '')).strip() for item in registry.get('models', [])]
            for model_id in sorted(discovered, key=_version_sort_key):
                _maybe_add_model(model_id)

        for model_id in BENCHMATCH_MODEL_IDS:
            _maybe_add_model(model_id)

        if BENCHMATCH_MODEL_LIMIT > 0:
            selected_model_ids = selected_model_ids[:BENCHMATCH_MODEL_LIMIT]

        bench_agents = list(BENCHMATCH_CLASSIC_AGENTS) + [f'model:{model_id}' for model_id in selected_model_ids]
        matchups = []
        if BENCHMATCH_ORDERED_MATCHUPS:
            for agent_a in bench_agents:
                for agent_b in bench_agents:
                    if not BENCHMATCH_INCLUDE_SELF_PLAY and agent_a == agent_b:
                        continue
                    matchups.append(f'{agent_a} vs {agent_b}')
        else:
            for idx, agent_a in enumerate(bench_agents):
                start = idx if BENCHMATCH_INCLUDE_SELF_PLAY else idx + 1
                for agent_b in bench_agents[start:]:
                    if not BENCHMATCH_INCLUDE_SELF_PLAY and agent_a == agent_b:
                        continue
                    matchups.append(f'{agent_a} vs {agent_b}')

        dataset_cfg['source_mode'] = 'benchmatch'
        dataset_cfg['dataset_source_id'] = DATASET_SOURCE_ID
        dataset_cfg['target_samples'] = int(TARGET_SAMPLES)
        dataset_cfg['games'] = int(BENCHMATCH_GAMES)
        dataset_cfg['matchups'] = matchups
        dataset_cfg['model_agent_device'] = str(BENCHMATCH_MODEL_AGENT_DEVICE)
        dataset_cfg['progress_update_every_n_games'] = 1
        cfg['dataset_generation'] = dataset_cfg

        target_path.parent.mkdir(parents=True, exist_ok=True)
        target_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
        DATASET_GENERATE_CONFIG_ACTIVE = str(target_path)

        print('Config benchmatch active prete:')
        print('  DATASET_GENERATE_CONFIG_ACTIVE =', DATASET_GENERATE_CONFIG_ACTIVE)
        print('  classic_agents                =', BENCHMATCH_CLASSIC_AGENTS)
        print('  selected_model_ids            =', selected_model_ids)
        print('  total_agents                  =', len(bench_agents))
        print('  total_matchups                =', len(matchups))
        print('  target_samples                =', TARGET_SAMPLES)


In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=$WORKTREE/src $PYTHON_BIN -m songo_model_stockfish.cli.main dataset-generate --config $DATASET_GENERATE_CONFIG_ACTIVE --job-id $DATASET_GENERATE_JOB_ID $DATASET_GENERATE_EXTRA_ARGS"


Lecture rapide du dernier resume `dataset-generate`, avec affichage detaille du `source_breakdown` quand le mode est `merge_existing`.

In [ ]:
from pathlib import Path
import json
import re

jobs_root = Path(DRIVE_ROOT) / 'jobs'
requested_job_id = DATASET_GENERATE_JOB_ID

match = re.match(r'^(.*?)(\d+)$', requested_job_id)
if match:
    prefix = match.group(1)
    candidates = [path for path in jobs_root.iterdir() if path.is_dir() and re.match(rf'^{re.escape(prefix)}\d+$', path.name)]
else:
    candidates = [path for path in jobs_root.iterdir() if path.is_dir() and path.name.startswith(requested_job_id)]

if not candidates:
    print('Aucun job dataset-generate trouve pour', requested_job_id)
else:
    latest_job_dir = sorted(candidates, key=lambda path: path.stat().st_mtime)[-1]
    summary_path = latest_job_dir / 'dataset_generation' / 'dataset_generation_summary.json'
    if not summary_path.exists():
        print('Resume introuvable:', summary_path)
    else:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        print('job_id               =', summary.get('job_id'))
        print('dataset_source_id    =', summary.get('dataset_source_id'))
        print('source_mode          =', summary.get('source_mode'))
        print('total_samples        =', summary.get('total_samples'))
        print('selected_samples     =', summary.get('selected_samples'))
        print('duplicate_samples    =', summary.get('duplicate_samples'))
        print('sampled_dir          =', summary.get('sampled_dir'))
        source_mode = summary.get('source_mode')
        source_breakdown = summary.get('source_breakdown', {})
        if source_breakdown:
            print('\nSource breakdown:')
            ranked_items = sorted(source_breakdown.items(), key=lambda item: item[1].get('selected_samples', 0), reverse=True)
            for source_id, stats in ranked_items:
                print(
                    '-', source_id,
                    '| scanned_files=', stats.get('scanned_files', 0),
                    '| scanned_samples=', stats.get('scanned_samples', 0),
                    '| selected_files=', stats.get('selected_files', 0),
                    '| selected_samples=', stats.get('selected_samples', 0),
                    '| duplicate_samples=', stats.get('duplicate_samples', 0),
                    '| copied_raw_files=', stats.get('copied_raw_files', 0),
                )
        elif source_mode == 'augment_existing':
            depth_breakdown = summary.get('augmentation_depth_breakdown', {})
            file_breakdown = summary.get('source_file_breakdown', {})
            if depth_breakdown:
                print('\nAugmentation depth breakdown:')
                for depth, count in sorted(depth_breakdown.items(), key=lambda item: int(item[0])):
                    print('  depth', depth, '=>', count)
            if file_breakdown:
                print('\nAugmentation source file breakdown:')
                ranked_files = sorted(file_breakdown.items(), key=lambda item: item[1].get('selected_augmented_samples', 0), reverse=True)
                for relative_name, stats in ranked_files[:25]:
                    print(
                        '-', relative_name,
                        '| scanned_samples=', stats.get('scanned_samples', 0),
                        '| selected_original=', stats.get('selected_original_samples', 0),
                        '| selected_augmented=', stats.get('selected_augmented_samples', 0),
                        '| duplicates=', stats.get('duplicate_samples', 0),
                        '| max_depth=', stats.get('max_depth_reached', 0),
                    )
            if not depth_breakdown and not file_breakdown:
                print('\nPas de breakdown d\'augmentation pour ce resume')
        else:
            print('\nPas de source_breakdown pour ce mode de generation')


## 8. Construire le dataset final

Le build exporte maintenant aussi des snapshots partiels du dataset final pendant le run. Des que `train.npz` et `validation.npz` existent dans le dossier du dataset courant, le training peut deja commencer a l'utiliser, sans attendre la fin complete du build.

In [10]:
!bash -lc "cd $WORKTREE && PYTHONPATH=$WORKTREE/src $PYTHON_BIN -m songo_model_stockfish.cli.main dataset-build --config $DATASET_BUILD_CONFIG --job-id $DATASET_BUILD_JOB_ID $DATASET_BUILD_EXTRA_ARGS"

/usr/bin/python3: Error while finding module specification for 'songo_model_stockfish.cli.main' (ModuleNotFoundError: No module named 'songo_model_stockfish')


## 9. Fusionner des datasets finaux existants

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=$WORKTREE/src $PYTHON_BIN -m songo_model_stockfish.cli.main dataset-merge-final --config $DATASET_MERGE_FINAL_CONFIG --job-id $DATASET_MERGE_FINAL_JOB_ID $DATASET_MERGE_FINAL_EXTRA_ARGS"

Lecture rapide du dernier resume `dataset-merge-final`, avec affichage du `source_breakdown` par split et par dataset final source.

In [ ]:
from pathlib import Path
import json
import re

jobs_root = Path(DRIVE_ROOT) / 'jobs'
requested_job_id = DATASET_MERGE_FINAL_JOB_ID

match = re.match(r'^(.*?)(\d+)$', requested_job_id)
if match:
    prefix = match.group(1)
    candidates = [path for path in jobs_root.iterdir() if path.is_dir() and re.match(rf'^{re.escape(prefix)}\d+$', path.name)]
else:
    candidates = [path for path in jobs_root.iterdir() if path.is_dir() and path.name.startswith(requested_job_id)]

if not candidates:
    print('Aucun job dataset-merge-final trouve pour', requested_job_id)
else:
    latest_job_dir = sorted(candidates, key=lambda path: path.stat().st_mtime)[-1]
    summary_path = latest_job_dir / 'dataset_merge_final' / 'dataset_merge_final_summary.json'
    if not summary_path.exists():
        print('Resume introuvable:', summary_path)
    else:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        print('job_id               =', summary.get('job_id'))
        print('dataset_id           =', summary.get('dataset_id'))
        print('build_mode           =', summary.get('build_mode'))
        print('teacher              =', f"{summary.get('teacher_engine')}:{summary.get('teacher_level')}")
        print('labeled_samples      =', summary.get('labeled_samples'))
        print('output_dir           =', summary.get('output_dir'))
        print('source_dataset_ids   =', summary.get('source_dataset_ids'))
        source_breakdown = summary.get('source_breakdown', {})
        if source_breakdown:
            for split_name, split_stats in source_breakdown.items():
                print(f'\nSplit: {split_name}')
                ranked_items = sorted(split_stats.items(), key=lambda item: item[1].get('kept_samples', 0), reverse=True)
                for source_id, stats in ranked_items:
                    print(
                        '-', source_id,
                        '| input_samples=', stats.get('input_samples', 0),
                        '| kept_samples=', stats.get('kept_samples', 0),
                        '| duplicate_samples=', stats.get('duplicate_samples', 0),
                        '| unique_games=', stats.get('unique_games', 0),
                    )
        else:
            print('\nPas de source_breakdown pour ce merge final')


## 10. Entrainement GPU

Previsualisation du dataset final le plus grand et du modele promu courant avant le training.

In [ ]:
from pathlib import Path
import json

registry_path = Path(DRIVE_ROOT) / 'data' / 'dataset_registry.json'
promoted_metadata_path = Path(DRIVE_ROOT) / 'models' / 'promoted' / 'best' / 'metadata.json'

if registry_path.exists():
    registry = json.loads(registry_path.read_text(encoding='utf-8'))
    candidates = []
    for item in registry.get('built_datasets', []):
        output_dir = Path(str(item.get('output_dir', '')))
        if (output_dir / 'train.npz').exists() and (output_dir / 'validation.npz').exists():
            candidates.append(item)
    if candidates:
        candidates = sorted(
            candidates,
            key=lambda item: (int(item.get('labeled_samples', 0)), str(item.get('updated_at', ''))),
            reverse=True,
        )
        best_dataset = candidates[0]
        print('Largest final dataset selected by training:')
        print('  dataset_id      =', best_dataset.get('dataset_id'))
        print('  build_mode      =', best_dataset.get('build_mode', 'teacher_label'))
        print('  labeled_samples =', best_dataset.get('labeled_samples'))
        print('  teacher         =', f"{best_dataset.get('teacher_engine')}:{best_dataset.get('teacher_level')}")
        print('  output_dir      =', best_dataset.get('output_dir'))
    else:
        print('Aucun built dataset complet avec train.npz + validation.npz trouve dans le registre')
else:
    print('dataset_registry.json introuvable:', registry_path)

if promoted_metadata_path.exists():
    promoted_metadata = json.loads(promoted_metadata_path.read_text(encoding='utf-8'))
    print('\nCurrent promoted best model:')
    print('  model_id               =', promoted_metadata.get('model_id'))
    print('  promoted_checkpoint    =', promoted_metadata.get('promoted_checkpoint_path'))
    print('  best_validation_metric =', promoted_metadata.get('best_validation_metric'))
    print('  benchmark_score        =', promoted_metadata.get('benchmark_score'))
else:
    print('\nAucun metadata de modele promu trouve:', promoted_metadata_path)


Comparer rapidement deux datasets finaux existants a partir du registre et de leur `dataset_metadata.json`.

In [ ]:
from pathlib import Path
import json

registry_path = Path(DRIVE_ROOT) / 'data' / 'dataset_registry.json'
if not COMPARE_DATASET_ID_A or not COMPARE_DATASET_ID_B:
    print('Renseigne COMPARE_DATASET_ID_A et COMPARE_DATASET_ID_B pour lancer la comparaison')
elif not registry_path.exists():
    print('Aucun dataset_registry.json trouve:', registry_path)
else:
    registry = json.loads(registry_path.read_text(encoding='utf-8'))
    built_index = {
        str(item.get('dataset_id')): item
        for item in registry.get('built_datasets', [])
        if str(item.get('dataset_id', '')).strip()
    }

    def _load_metadata(dataset_id: str):
        entry = built_index.get(dataset_id)
        if not entry:
            return None, f'Dataset introuvable dans le registre: {dataset_id}'
        metadata_path = Path(str(entry['output_dir'])) / 'dataset_metadata.json'
        if not metadata_path.exists():
            return None, f'dataset_metadata.json introuvable pour {dataset_id}: {metadata_path}'
        return json.loads(metadata_path.read_text(encoding='utf-8')), None

    metadata_a, error_a = _load_metadata(COMPARE_DATASET_ID_A)
    metadata_b, error_b = _load_metadata(COMPARE_DATASET_ID_B)
    if error_a:
        print(error_a)
    if error_b:
        print(error_b)
    if metadata_a and metadata_b:
        print('Dataset A =', COMPARE_DATASET_ID_A)
        print('  build_mode   =', metadata_a.get('build_mode'))
        print('  teacher      =', f"{metadata_a.get('teacher_engine')}:{metadata_a.get('teacher_level')}")
        print('  labeled      =', metadata_a.get('labeled_samples'))
        print('  source_ids   =', metadata_a.get('source_dataset_ids', []))
        print('  output_dir   =', metadata_a.get('output_dir'))
        print('Dataset B =', COMPARE_DATASET_ID_B)
        print('  build_mode   =', metadata_b.get('build_mode'))
        print('  teacher      =', f"{metadata_b.get('teacher_engine')}:{metadata_b.get('teacher_level')}")
        print('  labeled      =', metadata_b.get('labeled_samples'))
        print('  source_ids   =', metadata_b.get('source_dataset_ids', []))
        print('  output_dir   =', metadata_b.get('output_dir'))

        print('\nSplit comparison:')
        all_splits = sorted(set(metadata_a.get('splits', {}).keys()) | set(metadata_b.get('splits', {}).keys()))
        for split_name in all_splits:
            split_a = metadata_a.get('splits', {}).get(split_name, {})
            split_b = metadata_b.get('splits', {}).get(split_name, {})
            samples_a = int(split_a.get('samples', 0))
            samples_b = int(split_b.get('samples', 0))
            games_a = int(split_a.get('games', 0))
            games_b = int(split_b.get('games', 0))
            print(
                '-', split_name,
                '| A samples=', samples_a,
                '| B samples=', samples_b,
                '| diff=', samples_b - samples_a,
                '| A games=', games_a,
                '| B games=', games_b,
            )

        if metadata_a.get('teacher_engine') == metadata_b.get('teacher_engine') and metadata_a.get('teacher_level') == metadata_b.get('teacher_level'):
            print('\nTeacher compatible: oui')
        else:
            print('\nTeacher compatible: non')

        merge_breakdown_b = metadata_b.get('merge_breakdown') or {}
        if merge_breakdown_b:
            print('\nMerge breakdown for dataset B:')
            for split_name, split_stats in merge_breakdown_b.items():
                print(
                    '-', split_name,
                    '| input_samples=', split_stats.get('input_samples', 0),
                    '| kept_samples=', split_stats.get('kept_samples', 0),
                    '| duplicate_samples=', split_stats.get('duplicate_samples', 0),
                    '| unique_games=', split_stats.get('unique_games', 0),
                )


In [ ]:
# Training path 1: continue from the current promoted best model on the largest built dataset currently available (partial or completed)
!bash -lc "cd $WORKTREE && PYTHONPATH=$WORKTREE/src $PYTHON_BIN -m songo_model_stockfish.cli.main train --config $TRAIN_CONTINUE_CONFIG_ACTIVE --job-id $TRAIN_CONTINUE_JOB_ID"

In [ ]:
# Training path 2: start from scratch on the same largest built dataset currently available (partial or completed)
!bash -lc "cd $WORKTREE && PYTHONPATH=$WORKTREE/src $PYTHON_BIN -m songo_model_stockfish.cli.main train --config $TRAIN_SCRATCH_CONFIG_ACTIVE --job-id $TRAIN_SCRATCH_JOB_ID"

In [ ]:
from pathlib import Path

try:
    import yaml
except ModuleNotFoundError:
    yaml = None

if yaml is None:
    print('PyYAML introuvable, impossible de generer les configs partial-only automatiquement')
else:
    continue_base_path = Path(TRAIN_CONTINUE_CONFIG_ACTIVE)
    scratch_base_path = Path(TRAIN_SCRATCH_CONFIG_ACTIVE)
    if not continue_base_path.is_absolute():
        continue_base_path = Path(WORKTREE) / continue_base_path
    if not scratch_base_path.is_absolute():
        scratch_base_path = Path(WORKTREE) / scratch_base_path
    continue_partial_path = Path(TRAIN_CONTINUE_PARTIAL_CONFIG_PATH)
    scratch_partial_path = Path(TRAIN_SCRATCH_PARTIAL_CONFIG_PATH)

    def _write_partial_only_train_config(base_path: Path, target_path: Path):
        cfg = yaml.safe_load(base_path.read_text(encoding='utf-8')) or {}
        train_cfg = dict(cfg.get('train', {}) or {})
        train_cfg['dataset_selection_mode'] = 'configured'
        train_cfg['dataset_id'] = PARTIAL_TRAIN_DATASET_ID
        train_cfg['dataset_path'] = ''
        train_cfg['validation_path'] = ''
        cfg['train'] = train_cfg
        target_path.parent.mkdir(parents=True, exist_ok=True)
        target_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')

    _write_partial_only_train_config(continue_base_path, continue_partial_path)
    _write_partial_only_train_config(scratch_base_path, scratch_partial_path)

    print('Partial-only train configs generated:')
    print('  dataset_id                 =', PARTIAL_TRAIN_DATASET_ID)
    print('  continue_config_path       =', continue_partial_path)
    print('  scratch_config_path        =', scratch_partial_path)


In [ ]:
# Training path 3: continue from the current promoted best model on the selected partial dataset only
from pathlib import Path

try:
    import yaml
except ModuleNotFoundError:
    yaml = None

if yaml is None:
    print('PyYAML introuvable, impossible de generer la config partial-only automatiquement')
else:
    base_path = Path(TRAIN_CONTINUE_CONFIG_ACTIVE)
    if not base_path.is_absolute():
        base_path = Path(WORKTREE) / base_path
    target_path = Path(TRAIN_CONTINUE_PARTIAL_CONFIG_PATH)
    cfg = yaml.safe_load(base_path.read_text(encoding='utf-8')) or {}
    train_cfg = dict(cfg.get('train', {}) or {})
    train_cfg['dataset_selection_mode'] = 'configured'
    train_cfg['dataset_id'] = PARTIAL_TRAIN_DATASET_ID
    train_cfg['dataset_path'] = ''
    train_cfg['validation_path'] = ''
    cfg['train'] = train_cfg
    target_path.parent.mkdir(parents=True, exist_ok=True)
    target_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
    print('Config partial-only prete:', target_path)

!bash -lc "cd $WORKTREE && PYTHONPATH=$WORKTREE/src $PYTHON_BIN -m songo_model_stockfish.cli.main train --config $TRAIN_CONTINUE_PARTIAL_CONFIG_PATH --job-id $TRAIN_CONTINUE_PARTIAL_JOB_ID"

In [ ]:
# Training path 4: start from scratch on the selected partial dataset only
from pathlib import Path

try:
    import yaml
except ModuleNotFoundError:
    yaml = None

if yaml is None:
    print('PyYAML introuvable, impossible de generer la config partial-only automatiquement')
else:
    base_path = Path(TRAIN_SCRATCH_CONFIG_ACTIVE)
    if not base_path.is_absolute():
        base_path = Path(WORKTREE) / base_path
    target_path = Path(TRAIN_SCRATCH_PARTIAL_CONFIG_PATH)
    cfg = yaml.safe_load(base_path.read_text(encoding='utf-8')) or {}
    train_cfg = dict(cfg.get('train', {}) or {})
    train_cfg['dataset_selection_mode'] = 'configured'
    train_cfg['dataset_id'] = PARTIAL_TRAIN_DATASET_ID
    train_cfg['dataset_path'] = ''
    train_cfg['validation_path'] = ''
    cfg['train'] = train_cfg
    target_path.parent.mkdir(parents=True, exist_ok=True)
    target_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
    print('Config partial-only prete:', target_path)

!bash -lc "cd $WORKTREE && PYTHONPATH=$WORKTREE/src $PYTHON_BIN -m songo_model_stockfish.cli.main train --config $TRAIN_SCRATCH_PARTIAL_CONFIG_PATH --job-id $TRAIN_SCRATCH_PARTIAL_JOB_ID"

## 11. Evaluation

Previsualisation du dataset final le plus grand et du modele/checkpoint qui seront utilises par l'evaluation.

In [ ]:
from pathlib import Path
import json

registry_path = Path(DRIVE_ROOT) / 'data' / 'dataset_registry.json'
models_root = Path(DRIVE_ROOT) / 'models'
promoted_metadata_path = models_root / 'promoted' / 'best' / 'metadata.json'
model_registry_path = models_root / 'model_registry.json'

if registry_path.exists():
    registry = json.loads(registry_path.read_text(encoding='utf-8'))
    candidates = []
    for item in registry.get('built_datasets', []):
        output_dir = Path(str(item.get('output_dir', '')))
        if (output_dir / 'test.npz').exists():
            candidates.append(item)
    if candidates:
        candidates = sorted(
            candidates,
            key=lambda item: (int(item.get('labeled_samples', 0)), str(item.get('updated_at', ''))),
            reverse=True,
        )
        best_dataset = candidates[0]
        print('Largest final dataset selected by evaluation:')
        print('  dataset_id      =', best_dataset.get('dataset_id'))
        print('  build_mode      =', best_dataset.get('build_mode', 'teacher_label'))
        print('  labeled_samples =', best_dataset.get('labeled_samples'))
        print('  teacher         =', f"{best_dataset.get('teacher_engine')}:{best_dataset.get('teacher_level')}")
        print('  test_path       =', Path(str(best_dataset.get('output_dir', ''))) / 'test.npz')
    else:
        print('Aucun built dataset avec test.npz trouve dans le registre')
else:
    print('dataset_registry.json introuvable:', registry_path)

eval_model_mode = 'auto_latest'
if model_registry_path.exists():
    model_registry = json.loads(model_registry_path.read_text(encoding='utf-8'))
    models = model_registry.get('models', [])
    if models:
        models = sorted(models, key=lambda item: float(item.get('sort_ts', 0.0)), reverse=True)
        latest_model = models[0]
        print('\nCurrent latest model for evaluation auto_latest:')
        print('  model_id          =', latest_model.get('model_id'))
        print('  checkpoint_path   =', latest_model.get('checkpoint_path'))
        print('  evaluation_top1   =', latest_model.get('evaluation_top1'))
        print('  benchmark_score   =', latest_model.get('benchmark_score'))
    else:
        print('\nAucun modele present dans model_registry.json')
else:
    print('\nmodel_registry.json introuvable:', model_registry_path)

if promoted_metadata_path.exists():
    promoted_metadata = json.loads(promoted_metadata_path.read_text(encoding='utf-8'))
    print('\nCurrent promoted best model (utile si evaluation auto_best):')
    print('  model_id               =', promoted_metadata.get('model_id'))
    print('  promoted_checkpoint    =', promoted_metadata.get('promoted_checkpoint_path'))
    print('  best_validation_metric =', promoted_metadata.get('best_validation_metric'))
    print('  benchmark_score        =', promoted_metadata.get('benchmark_score'))


In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=$WORKTREE/src $PYTHON_BIN -m songo_model_stockfish.cli.main evaluate --config $EVALUATION_CONFIG_ACTIVE --job-id $EVALUATION_JOB_ID"

Lecture complete du dernier `evaluation_summary.json` pour le job d'evaluation courant.

In [9]:
from pathlib import Path
import json
import re

jobs_root = Path(DRIVE_ROOT) / 'jobs'
requested_job_id = globals().get('EVALUATION_JOB_ID', '')

if requested_job_id:
    match = re.match(r'^(.*?)(\d+)$', requested_job_id)
    if match:
        prefix = match.group(1)
        candidates = [path for path in jobs_root.iterdir() if path.is_dir() and re.match(rf'^{re.escape(prefix)}\d+$', path.name)]
    else:
        candidates = [path for path in jobs_root.iterdir() if path.is_dir() and path.name.startswith(requested_job_id)]
else:
    candidates = [path for path in jobs_root.iterdir() if path.is_dir() and path.name.startswith('eval_')]

if not candidates:
    print('Aucun job evaluation trouve', 'pour ' + requested_job_id if requested_job_id else '(aucun EVALUATION_JOB_ID defini et aucun job eval_ detecte)')
else:
    latest_job_dir = sorted(candidates, key=lambda path: path.stat().st_mtime)[-1]
    print('Evaluation job lu =', latest_job_dir.name)
    summary_path = latest_job_dir / 'evaluation_summary.json'
    if not summary_path.exists():
        print('Resume introuvable:', summary_path)
    else:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        print(json.dumps(summary, indent=2, ensure_ascii=True))


Evaluation job lu = eval_colab_pro_stable_007
{
  "job_id": "eval_colab_pro_stable_007",
  "model_id": "songo_policy_value_colab_pro_v11",
  "dataset_id": "dataset_merged_final_colab_pro_insane_1m",
  "dataset_selection_mode": "largest_built",
  "dataset_resolution": {
    "selection_mode": "largest_built",
    "resolved_from_registry": true,
    "selected_labeled_samples": 754447,
    "selected_build_mode": "merged_final",
    "selected_teacher_engine": "minimax",
    "selected_teacher_level": "insane",
    "selected_output_dir": "/content/drive/MyDrive/songo-stockfish/data/datasets/dataset_merged_final_colab_pro"
  },
  "device": "cpu",
  "mixed_precision": false,
  "examples": 80204,
  "loss_total": 2.525741659990303,
  "loss_policy": 1.0677553318033288,
  "loss_value": 0.07643517490280732,
  "value_mae": 0.15371281671069828,
  "policy_accuracy_top1": 0.6535833624258142,
  "policy_accuracy_top3": 0.9112637773677124,
  "checkpoint_path": "/content/drive/MyDrive/songo-stockfish/models

## 12. Benchmark

In [ ]:
from pathlib import Path
import json

try:
    import yaml
except ModuleNotFoundError:
    yaml = None

benchmark_config_path = Path(BENCHMARK_CONFIG_ACTIVE)
if not benchmark_config_path.is_absolute():
    benchmark_config_path = Path(WORKTREE) / benchmark_config_path
models_root = Path(DRIVE_ROOT) / 'models'
promoted_metadata_path = models_root / 'promoted' / 'best' / 'metadata.json'
model_registry_path = models_root / 'model_registry.json'

benchmark_cfg = {}
if yaml is None:
    print("PyYAML introuvable dans l'environnement notebook, impossible de lire la config benchmark automatiquement")
elif not benchmark_config_path.exists():
    print('Fichier de config benchmark introuvable:', benchmark_config_path)
else:
    raw_cfg = yaml.safe_load(benchmark_config_path.read_text(encoding='utf-8')) or {}
    benchmark_cfg = raw_cfg.get('benchmark', {}) or {}

target_mode = str(benchmark_cfg.get('target', 'auto_latest')) if benchmark_cfg else 'auto_latest'
checkpoint_cfg = str(benchmark_cfg.get('checkpoint_path', '')) if benchmark_cfg else ''
matchups = list(benchmark_cfg.get('matchups', [])) if benchmark_cfg else []
opponent_ratings = dict(benchmark_cfg.get('opponent_ratings', {})) if benchmark_cfg else {}
games_per_matchup = benchmark_cfg.get('games_per_matchup') if benchmark_cfg else None
alternate_first_player = benchmark_cfg.get('alternate_first_player') if benchmark_cfg else None
max_moves = benchmark_cfg.get('max_moves') if benchmark_cfg else None

print('Benchmark preview:')
print('  config_path             =', benchmark_config_path)
print('  target_mode             =', target_mode)
print('  checkpoint_path_cfg     =', checkpoint_cfg or '<auto>')
print('  matchups                =', ', '.join(matchups) if matchups else '<aucun>')
print('  games_per_matchup       =', games_per_matchup)
print('  alternate_first_player  =', alternate_first_player)
print('  max_moves               =', max_moves)
if opponent_ratings:
    print('  opponent_ratings        =')
    for key in matchups:
        print('    -', key, '=>', opponent_ratings.get(key, '<default>'))

latest_model = None
if model_registry_path.exists():
    model_registry = json.loads(model_registry_path.read_text(encoding='utf-8'))
    models = model_registry.get('models', [])
    if models:
        models = sorted(models, key=lambda item: float(item.get('sort_ts', 0.0)), reverse=True)
        latest_model = models[0]

if latest_model:
    print('\nCurrent latest model for benchmark auto_latest:')
    print('  model_id                =', latest_model.get('model_id'))
    print('  latest_checkpoint       =', latest_model.get('latest_checkpoint_path'))
    print('  benchmark_score         =', latest_model.get('benchmark_score'))
    print('  benchmark_elo_estimate  =', latest_model.get('benchmark_elo_estimate'))
    print('  benchmark_history_path  =', latest_model.get('benchmark_history_path'))
else:
    print('\nAucun modele disponible dans model_registry.json pour auto_latest')

promoted_metadata = None
if promoted_metadata_path.exists():
    promoted_metadata = json.loads(promoted_metadata_path.read_text(encoding='utf-8'))
    print('\nCurrent promoted best model:')
    print('  model_id               =', promoted_metadata.get('model_id'))
    print('  promoted_checkpoint    =', promoted_metadata.get('promoted_checkpoint_path'))
    print('  best_validation_metric =', promoted_metadata.get('best_validation_metric'))
    print('  benchmark_score        =', promoted_metadata.get('benchmark_score'))
else:
    print('\nAucun metadata de modele promu trouve:', promoted_metadata_path)

print('\nBenchmark target resolved preview:')
if target_mode in {'engine_v1'} or checkpoint_cfg == 'engine_v1':
    print('  resolved_target        = engine_v1')
elif target_mode in {'auto_best', 'auto_promoted_best'} or checkpoint_cfg in {'auto_best', 'auto_promoted_best'}:
    if promoted_metadata:
        print('  resolved_target        = auto_best / auto_promoted_best')
        print('  resolved_model_id      =', promoted_metadata.get('model_id'))
        print('  resolved_checkpoint    =', models_root / 'promoted' / 'best' / 'model.pt')
    else:
        print('  resolved_target        = auto_best / auto_promoted_best')
        print('  resolved_checkpoint    = indisponible, aucun modele promu trouve')
elif target_mode in {'auto', 'auto_latest'} or checkpoint_cfg in {'', 'auto', 'auto_latest'}:
    if latest_model:
        print('  resolved_target        = auto_latest')
        print('  resolved_model_id      =', latest_model.get('model_id'))
        print('  resolved_checkpoint    =', latest_model.get('latest_checkpoint_path'))
    else:
        print('  resolved_target        = auto_latest')
        print('  resolved_checkpoint    = indisponible, aucun modele dans le registre')
else:
    print('  resolved_target        = checkpoint explicite')
    print('  resolved_checkpoint    =', checkpoint_cfg)


In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=$WORKTREE/src $PYTHON_BIN -m songo_model_stockfish.cli.main benchmark --config $BENCHMARK_CONFIG_ACTIVE --job-id $BENCHMARK_JOB_ID"

In [ ]:
from pathlib import Path
import json
from datetime import datetime

reports_root = Path(DRIVE_ROOT) / 'reports' / 'benchmarks'
models_root = Path(DRIVE_ROOT) / 'models'
history_path = models_root / 'history' / 'benchmark_history.jsonl'
job_prefix = BENCHMARK_JOB_ID.rsplit('_v', 1)[0] if '_v' in BENCHMARK_JOB_ID else BENCHMARK_JOB_ID
job_dirs = sorted(
    [path for path in reports_root.glob(f'{job_prefix}*') if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

def _fmt_ts(value):
    if value in (None, ''):
        return '<unknown>'
    try:
        return datetime.fromtimestamp(float(value)).strftime('%Y-%m-%d %H:%M:%S')
    except Exception:
        return str(value)

if not job_dirs:
    print('Aucun job benchmark trouve pour le prefixe:', job_prefix)
else:
    selected_job_dir = job_dirs[0]
    summary_path = selected_job_dir / 'benchmark' / 'benchmark_summary.json'
    if not summary_path.exists():
        print('benchmark_summary.json introuvable:', summary_path)
    else:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        matchups = list(summary.get('matchups', []))
        print('Benchmark summary preview:')
        print('  job_dir                  =', selected_job_dir)
        print('  summary_path             =', summary_path)
        print('  engine                   =', summary.get('engine'))
        print('  matchups_count           =', len(matchups))
        print('  benchmark_score          =', summary.get('benchmark_score'))
        print('  benchmark_score_weighted =', summary.get('benchmark_score_weighted'))
        print('  benchmark_elo_estimate   =', summary.get('benchmark_elo_estimate'))
        print('  benchmark_history_path   =', summary.get('benchmark_history_path'))

        history_rows = []
        if history_path.exists() and summary.get('engine'):
            for line in history_path.read_text(encoding='utf-8').splitlines():
                line = line.strip()
                if not line:
                    continue
                payload = json.loads(line)
                if payload.get('model_id') == summary.get('engine'):
                    history_rows.append(payload)

        if history_rows:
            latest_history = history_rows[-1]
            print('\nLatest benchmark history entry for this model:')
            print('  model_id                =', latest_history.get('model_id'))
            print('  benchmark_score         =', latest_history.get('benchmark_score'))
            print('  benchmark_score_weighted=', latest_history.get('benchmark_score_weighted'))
            print('  benchmark_elo_estimate  =', latest_history.get('benchmark_elo_estimate'))
            print('  job_id                  =', latest_history.get('job_id'))
            print('  benchmark_summary_path  =', latest_history.get('benchmark_summary_path'))

            print('\nRecent benchmark history (last 8 runs):')
            print('  created_at           | job_id                       | weighted  | elo     | score')
            print('  ------------------- | ---------------------------- | --------- | ------- | -------')
            for row in history_rows[-8:]:
                created_at = _fmt_ts(row.get('created_at'))
                job_id = str(row.get('job_id', ''))[:28].ljust(28)
                weighted = f"{float(row.get('benchmark_score_weighted', -1.0)):.4f}"
                elo = row.get('benchmark_elo_estimate')
                elo_str = f"{float(elo):.1f}" if elo not in (None, '') else '<na>'
                score = f"{float(row.get('benchmark_score', -1.0)):.4f}"
                print(f'  {created_at} | {job_id} | {weighted:>9} | {elo_str:>7} | {score:>7}')

        if matchups:
            sorted_matchups = sorted(matchups, key=lambda item: float(item.get('winrate', 0.0)), reverse=True)
            avg_winrate = sum(float(item.get('winrate', 0.0)) for item in sorted_matchups) / len(sorted_matchups)
            total_games = sum(int(item.get('games', 0)) for item in sorted_matchups)
            total_wins = sum(int(item.get('wins_a', 0)) for item in sorted_matchups)
            total_losses = sum(int(item.get('wins_b', 0)) for item in sorted_matchups)
            total_draws = sum(int(item.get('draws', 0)) for item in sorted_matchups)
            print('  average_winrate          =', round(avg_winrate, 4))
            print('  total_games              =', total_games)
            print('  total_record             =', f'{total_wins}W / {total_losses}L / {total_draws}D')
            print('\nPer-matchup results:')
            for item in sorted_matchups:
                opponent = item.get('opponent')
                games = int(item.get('games', 0))
                wins = int(item.get('wins_a', 0))
                losses = int(item.get('wins_b', 0))
                draws = int(item.get('draws', 0))
                winrate = float(item.get('winrate', 0.0))
                score_rate = float(item.get('score_rate', 0.0))
                avg_moves = float(item.get('avg_moves', 0.0))
                weighted = winrate * float(item.get('difficulty_weight', 1.0))
                opp_rating = float(item.get('opponent_rating', 0.0))
                as_first = item.get('as_first_player', {})
                as_second = item.get('as_second_player', {})
                print(
                    f'  - {opponent}: {wins}W / {losses}L / {draws}D | games={games} | winrate={winrate:.4f} | score_rate={score_rate:.4f} | weighted={weighted:.4f} | opp_rating={opp_rating:.0f} | avg_moves={avg_moves:.1f}'
                )
                print(
                    f"    as_first={as_first.get('wins', 0)}W/{as_first.get('losses', 0)}L/{as_first.get('draws', 0)}D | winrate={float(as_first.get('winrate', 0.0)):.4f} | score_rate={float(as_first.get('score_rate', 0.0)):.4f}"
                )
                print(
                    f"    as_second={as_second.get('wins', 0)}W/{as_second.get('losses', 0)}L/{as_second.get('draws', 0)}D | winrate={float(as_second.get('winrate', 0.0)):.4f} | score_rate={float(as_second.get('score_rate', 0.0)):.4f}"
                )
        else:
            print('Aucun matchup present dans le summary benchmark')


In [ ]:
from pathlib import Path
import json
from datetime import datetime

models_root = Path(DRIVE_ROOT) / 'models'
model_registry_path = models_root / 'model_registry.json'
history_path = models_root / 'history' / 'benchmark_history.jsonl'

def _fmt_ts(value):
    if value in (None, ''):
        return '<unknown>'
    try:
        return datetime.fromtimestamp(float(value)).strftime('%Y-%m-%d %H:%M:%S')
    except Exception:
        return str(value)

def _load_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

if not model_registry_path.exists():
    print('model_registry.json introuvable:', model_registry_path)
else:
    registry = json.loads(model_registry_path.read_text(encoding='utf-8'))
    models = registry.get('models', [])
    if not models:
        print('Aucun modele present dans le registre')
    else:
        models = sorted(models, key=lambda item: float(item.get('sort_ts', 0.0)), reverse=True)
        selected_model_id = str(SELECTED_MODEL_ID).strip() if 'SELECTED_MODEL_ID' in globals() else ''
        current = None
        if selected_model_id:
            current = next((item for item in models if str(item.get('model_id', '')) == selected_model_id), None)
            if current is None:
                print(f"Modele demande introuvable dans le registre, fallback sur le dernier modele: {selected_model_id}")
        if current is None:
            current = models[0]
        model_id = str(current.get('model_id', ''))
        print('Unified latest model cycle view:')
        print('  selected_model_id        =', selected_model_id or '<latest>')
        print('  active_model_id          =', model_id)
        print('  training_job_id          =', current.get('training_job_id'))
        print('  best_validation_metric   =', current.get('best_validation_metric'))
        print('  evaluation_top1          =', current.get('evaluation_top1'))
        print('  benchmark_score          =', current.get('benchmark_score'))
        print('  benchmark_elo_estimate   =', current.get('benchmark_elo_estimate'))
        print('  model_card_path          =', current.get('model_card_path'))

        other_models = [item for item in models if str(item.get('model_id', '')) != model_id]
        if other_models:
            print('\nOther available models in registry:')
            for idx, item in enumerate(other_models[:8], start=1):
                print(
                    f"  {idx}. {item.get('model_id')} | best_val={item.get('best_validation_metric')} | eval_top1={item.get('evaluation_top1')} | bench={item.get('benchmark_score')} | elo={item.get('benchmark_elo_estimate')}"
                )

        model_card_path = Path(str(current.get('model_card_path', ''))) if current.get('model_card_path') else None
        model_card = _load_json(model_card_path) if model_card_path else None

        training_summary = None
        evaluation_summary = None
        if model_card:
            training_job_id = str(model_card.get('training_job_id', '')).strip()
            if training_job_id:
                training_summary = _load_json(Path(DRIVE_ROOT) / 'jobs' / training_job_id / 'training_summary.json')
            eval_summary_path = str(model_card.get('evaluation_summary_path', '')).strip()
            if eval_summary_path:
                evaluation_summary = _load_json(Path(eval_summary_path))

        if training_summary:
            print('\nTraining snapshot:')
            print('  dataset_id               =', training_summary.get('dataset_id'))
            print('  selection_mode           =', training_summary.get('dataset_selection_mode'))
            print('  best_epoch               =', training_summary.get('best_epoch'))
            print('  best_validation_metric   =', training_summary.get('best_validation_metric'))
            print('  early_stopped            =', training_summary.get('early_stopped'))
            print('  final_model_path         =', training_summary.get('final_model_path'))
        else:
            print('\nTraining snapshot: introuvable')

        if evaluation_summary:
            print('\nLatest evaluation snapshot:')
            print('  dataset_id               =', evaluation_summary.get('dataset_id'))
            print('  selection_mode           =', evaluation_summary.get('dataset_selection_mode'))
            print('  policy_accuracy_top1     =', evaluation_summary.get('policy_accuracy_top1'))
            print('  policy_accuracy_top3     =', evaluation_summary.get('policy_accuracy_top3'))
            print('  value_mae                =', evaluation_summary.get('value_mae'))
            print('  loss_total               =', evaluation_summary.get('loss_total'))
            print('  evaluation_summary_path  =', model_card.get('evaluation_summary_path'))
        else:
            print('\nLatest evaluation snapshot: introuvable')

        history_rows = []
        if history_path.exists():
            for line in history_path.read_text(encoding='utf-8').splitlines():
                line = line.strip()
                if not line:
                    continue
                payload = json.loads(line)
                if payload.get('model_id') == model_id:
                    history_rows.append(payload)

        if history_rows:
            print('\nUnified recent cycle table (last 8 benchmark cycles for active model):')
            print('  created_at           | job_id                       | weighted  | elo     | eval_top1 | best_val')
            print('  ------------------- | ---------------------------- | --------- | ------- | --------- | --------')
            eval_top1 = current.get('evaluation_top1')
            best_val = current.get('best_validation_metric')
            eval_str = f"{float(eval_top1):.4f}" if eval_top1 not in (None, '', -1.0) else '<na>'
            best_val_str = f"{float(best_val):.4f}" if best_val not in (None, '', -1.0) else '<na>'
            for row in history_rows[-8:]:
                created_at = _fmt_ts(row.get('created_at'))
                job_id = str(row.get('job_id', ''))[:28].ljust(28)
                weighted = f"{float(row.get('benchmark_score_weighted', -1.0)):.4f}"
                elo = row.get('benchmark_elo_estimate')
                elo_str = f"{float(elo):.1f}" if elo not in (None, '') else '<na>'
                print(f'  {created_at} | {job_id} | {weighted:>9} | {elo_str:>7} | {eval_str:>9} | {best_val_str:>8}')
        else:
            print('\nUnified recent cycle table: aucun historique benchmark trouve pour ce modele')


## 11bis. Lire les resultats de comparaison

Ordre de lecture recommande:
- 1. `benchmark_score` et winrates benchmark
- 2. `policy_accuracy_top1`, `policy_accuracy_top3`, `value_mae`
- 3. `best_validation_metric`

Regle simple:
- si `residual` est meilleur en benchmark et pas moins bon en evaluation, il gagne
- s'il est meilleur en evaluation mais pas en benchmark, garde `stable`

Fichiers a comparer pour chaque variante:
- `jobs/<train_job_id>/training_summary.json`
- `jobs/<eval_job_id>/evaluation_summary.json`
- `jobs/<benchmark_job_id>/benchmark_summary.json`


## 12. Suivre un job en direct

In [ ]:
WATCH_JOB_ID = TRAIN_JOB_ID
!tail -f {DRIVE_ROOT}/jobs/{WATCH_JOB_ID}/events.jsonl

## 13. Reprendre un job interrompu

Relance simplement la meme cellule de commande avec le meme `job_id`.